# Fine-tuning Llama 3.1 8B for SOP Generation

This notebook fine-tunes Llama 3.1 8B using QLoRA (4-bit quantization) for efficient training on limited GPU resources.

**Training approach:**
- Base model: Llama 3.1 8B Instruct
- Method: QLoRA (Low-Rank Adaptation with 4-bit quantization)
- Dataset: 400 training + 100 validation SOPs

In [1]:
import torch

if torch.cuda.is_available():
    print(f"✓ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✓ Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("✗ No GPU!")

✓ GPU: Tesla T4
✓ Memory: 15.64 GB


## Install Required Libraries

Install HuggingFace libraries for fine-tuning:
- `transformers`: Model loading and training
- `peft`: Parameter-Efficient Fine-Tuning (QLoRA)
- `bitsandbytes`: 4-bit quantization
- `accelerate`: Distributed training utilities

In [2]:
# Install dependencies (minimal, stable)
!pip install -q \
  transformers==4.45.0 \
  accelerate==0.34.0 \
  peft==0.13.0 \
  datasets==2.21.0 \


print("✓ Dependencies installed!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 77.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.3/324.3 kB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.5/322.5 kB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 41.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 79.1 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.7/146.7 kB 14.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.26.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not ins

In [3]:
import transformers, accelerate, peft, datasets,huggingface_hub
print("transformers:", transformers.__version__)
print("accelerate:", accelerate.__version__)
print("peft:", peft.__version__)
print("datasets:", datasets.__version__)
print("huggingface_hub:", huggingface_hub.__version__)


transformers: 4.45.0
accelerate: 0.34.0
peft: 0.13.0
datasets: 2.21.0
huggingface_hub: 0.36.0


## Upload Training Data

Upload the JSONL files we prepared earlier.

In [6]:
import os

train_path = None
val_path = None

for root, _, files in os.walk("/kaggle/input"):
    if "train.jsonl" in files and train_path is None:
        train_path = os.path.join(root, "train.jsonl")
    if "val.jsonl" in files and val_path is None:
        val_path = os.path.join(root, "val.jsonl")

print("train.jsonl:", train_path)
print("val.jsonl:", val_path)


train.jsonl: /kaggle/input/sop-training-data/train.jsonl
val.jsonl: /kaggle/input/sop-training-data/val.jsonl


## Load Dataset

Load the JSONL files into HuggingFace Dataset format.

In [4]:
from datasets import load_dataset

train_dataset = load_dataset(
    'json', 
    data_files='/kaggle/input/sop-training-data/train.jsonl',
    split='train'
)

val_dataset = load_dataset(
    'json',
    data_files='/kaggle/input/sop-training-data/val.jsonl', 
    split='train'
)

print(f"✓ Training samples: {len(train_dataset)}")
print(f"✓ Validation samples: {len(val_dataset)}")
print("Columns:", train_dataset.column_names)
print("Sample keys:", train_dataset[0].keys())

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

✓ Training samples: 400
✓ Validation samples: 100
Columns: ['messages']
Sample keys: dict_keys(['messages'])


## 🔐 Hugging Face Login

Required to access gated models (e.g. Meta LLaMA).


In [5]:
from huggingface_hub import login
login()

## load Base Model

This cell loads the **LLaMA 3.2 (1B, Instruct)** model and its tokenizer.


In [6]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

MODEL_NAME = "meta-llama/Llama-3.2-1B-Instruct"

print(f"Loading {MODEL_NAME}...")
print("This may take 2-3 minutes...\n")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("✓ Model loaded!")
print(f"✓ Size: {model.get_memory_footprint() / 1e9:.2f} GB")

Loading meta-llama/Llama-3.2-1B-Instruct...
This may take 2-3 minutes...



config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

✓ Model loaded!
✓ Size: 2.47 GB


## Configure LoRA Adapters

Setup `LoRA` (Low-Rank Adaptation) for parameter-efficient fine-tuning:
- Only trains small adapter layers (~1% of parameters)
- Much faster and memory-efficient
- Achieves similar quality to full fine-tuning

In [7]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,                      # Rank of adaptation matrices
    lora_alpha=32,             # Scaling factor
    target_modules=[           # Layers to apply LoRA
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# Apply LoRA to model
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 3,407,872 || all params: 1,239,222,272 || trainable%: 0.2750


## Training Configuration

Define training hyperparameters:
- Batch size, learning rate, epochs
- Gradient accumulation for effective larger batches
- Evaluation and saving strategy

In [8]:
# Training arguments
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./sop-finetuned",
    num_train_epochs=5,  
    per_device_train_batch_size=2,  
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,  
    lr_scheduler_type="cosine",
    warmup_steps=100, 
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    fp16=True,
    report_to="none",
    remove_unused_columns=False,
)

print("✓ Training configuration ready (optimized for 1B)!")
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  Batch size: {training_args.per_device_train_batch_size}")
print(f"  Effective batch: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")

✓ Training configuration ready (optimized for 1B)!
  Epochs: 5
  Batch size: 2
  Effective batch: 8


## Tokenize with Assistant-Only Loss

In this step, the training data is converted into token IDs for model training.

- Only the **assistant’s response (SOP text)** is used to compute the training loss

- Tokens belonging to **system and user prompts** are masked with `-100`

- This improves instruction-following behavior and generation quality

In [9]:
# Tokenize data so loss is computed ONLY on the assistant answer (SOP text)
MAX_LEN = 1024

def tokenize_assistant_only(example):
    messages = example["messages"]

    # Separate assistant message from system/user messages
    assistant_msg = None
    non_assistant = []
    for m in messages:
        if m["role"] == "assistant":
            assistant_msg = m
        else:
            non_assistant.append(m)

    if assistant_msg is None:
        raise ValueError("No assistant message found in this example")

    # Prompt text (system+user) with a generation cue for assistant
    prompt_text = tokenizer.apply_chat_template(
        non_assistant,
        tokenize=False,
        add_generation_prompt=True
    )

    # Full conversation (system+user+assistant)
    full_text = tokenizer.apply_chat_template(
        non_assistant + [assistant_msg],
        tokenize=False,
        add_generation_prompt=False
    )

    # Tokenize prompt and full text
    prompt_ids = tokenizer(prompt_text, truncation=True, max_length=MAX_LEN, add_special_tokens=False)["input_ids"]
    full = tokenizer(full_text, truncation=True, max_length=MAX_LEN, add_special_tokens=False)

    input_ids = full["input_ids"]
    attention_mask = full["attention_mask"]

    # Build labels: mask prompt tokens with -100, keep assistant tokens for loss
    labels = input_ids.copy()
    prompt_len = min(len(prompt_ids), len(labels))
    labels[:prompt_len] = [-100] * prompt_len

    return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}

print("Tokenizing (assistant-only loss)...")

train_tok = train_dataset.map(tokenize_assistant_only, remove_columns=train_dataset.column_names)
val_tok   = val_dataset.map(tokenize_assistant_only, remove_columns=val_dataset.column_names)

print("✓ Tokenized")
print("Columns:", train_tok.column_names)
print("Example lengths:", len(train_tok[0]["input_ids"]), len(train_tok[0]["labels"]))
print("Masked tokens (-100) in labels:", sum(1 for x in train_tok[0]["labels"] if x == -100))


Tokenizing (assistant-only loss)...


Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

✓ Tokenized
Columns: ['input_ids', 'attention_mask', 'labels']
Example lengths: 723 723
Masked tokens (-100) in labels: 67


This data collator pads input sequences to the same length and aligns labels accordingly. Label padding uses `-100` so that padded tokens are ignored during loss computation, ensuring correct training for causal language models.


In [11]:
class CausalLMCollator:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def __call__(self, features):
        labels = [f["labels"] for f in features]
        for f in features:
            f.pop("labels")

        batch = self.tokenizer.pad(features, padding=True, return_tensors="pt")

        max_len = batch["input_ids"].shape[1]
        padded_labels = []
        for lab in labels:
            lab = lab[:max_len] + [-100] * (max_len - len(lab))
            padded_labels.append(lab)

        batch["labels"] = torch.tensor(padded_labels, dtype=torch.long)
        return batch

data_collator = CausalLMCollator(tokenizer)

## Initialize Trainer

Setup the SFTTrainer (Supervised Fine-Tuning Trainer) which handles:
- Training loop
- Evaluation
- Checkpointing
- Gradient updates

In [12]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    data_collator=data_collator,
)

print("✓ Trainer ready")

2026-02-06 14:32:27.472500: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770388347.653242      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770388347.702950      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770388348.142450      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770388348.142476      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770388348.142479      55 computation_placer.cc:177] computation placer alr

✓ Trainer ready


/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:494: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


## Start Training

Begin fine-tuning process. 

Progress will be shown below with loss values.

In [13]:
print("="*60)
print("STARTING FINE-TUNING")
print("="*60)
print("Monitor GPU usage and loss below\n")

trainer.train()

print("\n" + "="*60)
print("✓ TRAINING COMPLETE!")
print("="*60)

You're using a PreTrainedTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


STARTING FINE-TUNING
Monitor GPU usage and loss below



Step,Training Loss,Validation Loss
50,2.177000,2.327541
100,2.029100,2.237045
150,2.055200,2.146852
200,1.945200,2.088571
250,1.864000,2.082453



✓ TRAINING COMPLETE!


In [14]:
SAVE_DIR = "/kaggle/working/sop_lora_adapter"
trainer.model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

print("✓ LoRA adapter saved")
!ls -lh /kaggle/working/sop_lora_adapter


✓ LoRA adapter saved
total 30M
-rw-r--r-- 1 root root  686 Feb  6 14:45 adapter_config.json
-rw-r--r-- 1 root root  14M Feb  6 14:45 adapter_model.safetensors
-rw-r--r-- 1 root root 5.0K Feb  6 14:45 README.md
-rw-r--r-- 1 root root  325 Feb  6 14:45 special_tokens_map.json
-rw-r--r-- 1 root root  54K Feb  6 14:45 tokenizer_config.json
-rw-r--r-- 1 root root  17M Feb  6 14:45 tokenizer.json


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [15]:
!zip -r /kaggle/working/sop_lora_adapter.zip /kaggle/working/sop_lora_adapter
print("✓ Zipped and safe")


  adding: kaggle/working/sop_lora_adapter/ (stored 0%)
  adding: kaggle/working/sop_lora_adapter/tokenizer_config.json (deflated 94%)
  adding: kaggle/working/sop_lora_adapter/README.md (deflated 66%)
  adding: kaggle/working/sop_lora_adapter/tokenizer.json

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


 (deflated 85%)
  adding: kaggle/working/sop_lora_adapter/adapter_model.safetensors (deflated 8%)
  adding: kaggle/working/sop_lora_adapter/adapter_config.json (deflated 52%)
  adding: kaggle/working/sop_lora_adapter/special_tokens_map.json (deflated 63%)
✓ Zipped and safe


# test the model

In [39]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

BASE_MODEL = "meta-llama/Llama-3.2-1B-Instruct"
ADAPTER_PATH = "/kaggle/working/sop_lora_adapter"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="auto",
)

model = PeftModel.from_pretrained(model, ADAPTER_PATH)
model.eval()


# Test the fine-tuned model
print("="*60)
print("TESTING FINE-TUNED MODEL")
print("="*60)

# Test prompt
test_prompt = """<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are an expert at writing Statements of Purpose.<|eot_id|><|start_header_id|>user<|end_header_id|>

Write a Statement of Purpose for a Master's program in Computer Science.<|eot_id|><|start_header_id|>assistant<|end_header_id|>

"""

# Generate
inputs = tokenizer(test_prompt, return_tensors="pt").to(model.device)
outputs = model.generate(
    **inputs,
    max_new_tokens=850,
    temperature=0.7,
    top_p=0.9,
    repetition_penalty=1.15,
    do_sample=True
)

# Decode and print
generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
sop_only = generated_text.split("assistant<|end_header_id|>")[-1].strip()

print("\n" + "="*60)
print("GENERATED SOP:")
print("="*60)
print(sop_only)
print("="*60)

Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


TESTING FINE-TUNED MODEL

GENERATED SOP:
system

You are an expert at writing Statements of Purpose.user

Write a Statement of Purpose for a Master's program in Computer Science.assistant

**Statement of Purpose**

As I stand before the vast expanse of my future, I am driven by a singular passion: innovation and transformation through technology. My journey into computer science is not merely about seeking knowledge; it is an endeavor to harness this field as a tool to uplift society and address pressing global challenges.

Growing up in [Your City], I was always fascinated by how things work. From watching my parents tinker with their old electronics to exploring the intricacies of coding on family computers during weekends, I found myself captivated by the magic of computation. This curiosity led me to pursue engineering from [Undergraduate Institution]. Through rigorous academic coursework, I gained a solid foundation in both theoretical concepts and practical applications, but it w